# typical MLP

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error


import random

seed = 50

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==========================
# 1. data
# ==========================
data = np.load("..\data\hhg_dataset_SFA_HHG_Laserparam_15cycles.npz")

HHG_spec = data["HHG_spec"]
laser_param = data["y"]   # [intensity, sinCEP, cosCEP]
# --- HHG---
X = HHG_spec[:, 400:3400:3]     #
X = np.log1p(X)              # log

# normalization
X = X / np.max(X, axis=1, keepdims=True)

# laser param
y = laser_param[:, 0:3]      # intensity, sinCEP, cosCEP

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=seed
)

# StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)

# tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

#train_loader = DataLoader(
#    TensorDataset(X_train, y_train),
#    batch_size=32,
#    shuffle=True
#)

g = torch.Generator()
g.manual_seed(seed)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True,
    generator=g
)

# ==========================
# 2. typycal MLP model
# ==========================
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.mlp(x)
        
class AttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Attention
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, input_dim),
            #nn.Softmax(dim=1)
            nn.Sigmoid() 
        )

        # MLP
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            #nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        attn = self.attention(x)
#        x = x * (1.0 + 0.5 * attn)
        x = x * attn
#        x = x * (1.0 + attn)
        return self.mlp(x)

#model = AttentionMLP(input_dim=X_train.shape[1])
model = MLP(input_dim=X_train.shape[1])

optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
criterion = nn.MSELoss()

# ==========================
# 3. training
# ==========================

for epoch in range(2000):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)

        #
        loss = (
            1.0 * F.mse_loss(pred[:,0], yb[:,0]) +
            1.0 * F.mse_loss(pred[:,1], yb[:,1]) +
            1.0 * F.mse_loss(pred[:,2], yb[:,2])
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss {total_loss:.4f}")

# ==========================
# 4. eval
# ==========================

model.eval()
with torch.no_grad():
    pred_test = model(X_test)

# inversed transform
pred_test = scaler_y.inverse_transform(pred_test.numpy())
y_test_inv = scaler_y.inverse_transform(y_test.numpy())

# R2
print("=== R2 ===")
print("Intensity R2:", r2_score(y_test_inv[:,0], pred_test[:,0]))
print("sin CEP R2:", r2_score(y_test_inv[:,1], pred_test[:,1]))
print("cos CEP R2:", r2_score(y_test_inv[:,2], pred_test[:,2]))

# MAE
print("\n=== MAE (rad) ===")
print("sin:", mean_absolute_error(y_test_inv[:,1], pred_test[:,1]))
print("cos:", mean_absolute_error(y_test_inv[:,2], pred_test[:,2]))

cep_true = np.arctan2(y_test_inv[:,1], y_test_inv[:,2])
cep_pred1 = np.arctan2(pred_test[:,1], pred_test[:,2])


dcep1 = np.angle(np.exp(1j*(cep_pred1 - cep_true)))
dcep2 = np.angle(np.exp(1j * ((cep_pred1 - np.pi) - cep_true)))

# comparison between 0 and pi-shift
dcep = np.where(np.abs(dcep1) < np.abs(dcep2), dcep1, dcep2)
#CEP_pred2sin=reconstruct_cep_from_sin_cos(y_pred1[:,0], y_pred1[:,1])
#dcep2 = np.angle(np.exp(1j*(cep_pred2 - cep_true)))
#dcep2sin = np.angle(np.exp(1j*(CEP_pred2sin - cep_true )))

print("MAE(rad) =", np.mean(np.abs(dcep)))
print("MAE(deg) =", np.mean(np.abs(dcep))*180/np.pi)

# save model

In [ ]:
torch.save(model.state_dict(), "../models/MLP_seed50_n15cycle_lossMSE.pth")

# load model

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error


import random

seed = 0

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==========================
# 1. data
# ==========================
data = np.load("..\data\hhg_dataset_SFA_HHG_Laserparam_15cycles.npz")

HHG_spec = data["HHG_spec"]
laser_param = data["y"]   # [intensity, sinCEP, cosCEP]
# --- HHG ---
X = HHG_spec[:, 400:3400:3]     # 
X = np.log1p(X)              # 

# normalized
X = X / np.max(X, axis=1, keepdims=True)

# 出力
y = laser_param[:, 0:3]      # intensity, sinCEP, cosCEP

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=seed
)

# StandardScaler（重要）
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)

# tensor化
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

model.load_state_dict(torch.load("..\models\MLP_seed0_n15cycle_lossMSE.pth"))
model.eval()   # 推論前に呼ぶのが慣例


with torch.no_grad():
    pred_test = model(X_test)

# 逆スケーリング
pred_test = scaler_y.inverse_transform(pred_test.numpy())
y_test_inv = scaler_y.inverse_transform(y_test.numpy())

# R2
print("=== R2 ===")
print("Intensity R2:", r2_score(y_test_inv[:,0], pred_test[:,0]))
print("sin CEP R2:", r2_score(y_test_inv[:,1], pred_test[:,1]))
print("cos CEP R2:", r2_score(y_test_inv[:,2], pred_test[:,2]))

# MAE
print("\n=== MAE (rad) ===")
print("sin:", mean_absolute_error(y_test_inv[:,1], pred_test[:,1]))
print("cos:", mean_absolute_error(y_test_inv[:,2], pred_test[:,2]))

cep_true = np.arctan2(y_test_inv[:,1], y_test_inv[:,2])
cep_pred1 = np.arctan2(pred_test[:,1], pred_test[:,2])


dcep1 = np.angle(np.exp(1j*(cep_pred1 - cep_true)))
dcep2 = np.angle(np.exp(1j * ((cep_pred1 - np.pi) - cep_true)))

# comparison between 0 and pi-shift
dcep = np.where(np.abs(dcep1) < np.abs(dcep2), dcep1, dcep2)
#CEP_pred2sin=reconstruct_cep_from_sin_cos(y_pred1[:,0], y_pred1[:,1])
#dcep2 = np.angle(np.exp(1j*(cep_pred2 - cep_true)))
#dcep2sin = np.angle(np.exp(1j*(CEP_pred2sin - cep_true )))

print("MAE(rad) =", np.mean(np.abs(dcep)))
print("MAE(deg) =", np.mean(np.abs(dcep))*180/np.pi)